'''
Source: Impact4cast (Max Planck Institute)
Original code from the Max Planck Institute for Informatics / Max Planck Institute for Security and Privacy research group.

Modifications: Bug fixes for checkpoint resume mechanism and dataset adaptation for scientific entity impact prediction.
'''


Merge concept co-occurrence edge data scattered across multiple gz files into a unified dataset.Readconcept_pair all chunk edge data files in the folder, skip empty filesMerge items columns [description] Save as itemsall_concept_pairs.gz file, while recording detailed logs of the entire merge process.

In [ ]:
import gzip
import os
import time
from datetime import datetime
import pickle
import gc
import sys
import re

# ========== Configuration ==========
SOURCE_FOLDER = r"./data/entities_pair"
OUTPUT_FILE =r"./data/output\merged_edges.pkl.gz" 
LOG_FOLDER = r"./logs"

# Processing parameters
BATCH_SIZE = 100  # Code comment
MAX_EDGES_PER_CHUNK = 50000  # Code comment
PICKLE_PROTOCOL = pickle.HIGHEST_PROTOCOL  # Code comment

# ========== Initialization ==========
if not os.path.exists(LOG_FOLDER):
    os.makedirs(LOG_FOLDER)

log_file = os.path.join(LOG_FOLDER, f'log_merge_concept_pairs_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt')

def log_message(message, to_console=True, to_file=True):
    """..."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    full_message = f"[{timestamp}] {message}"
    
    if to_console:
        print(full_message)
    
    if to_file:
        with open(log_file, 'a', encoding='utf-8') as f:
            f.write(full_message + '\n')

def get_gz_files():
    """...edge_part_XXX.gz...Sort"""
    pattern = re.compile(r'edge_part_(\d+)\.gz$')
    files = []
    
    for filename in os.listdir(SOURCE_FOLDER):
        match = pattern.match(filename)
        if match:
            file_number = int(match.group(1))
            filepath = os.path.join(SOURCE_FOLDER, filename)
            files.append((file_number, filename, filepath))
    
    # Code comment
    files.sort(key=lambda x: x[0])
    
    log_message(f"Found {len(files)}  itemsedge_part_XXX.gz...")
    if len(files) > 0:
        log_message(f"...: {files[0][0]} - {files[-1][0]}")
    
    return files

def load_gz_file(filepath, filename=""):
    """...Loadgz...pickle..."""
    try:
        file_size = os.path.getsize(filepath)
        if file_size == 0:
            return None, f"Empty file: {file_size}bytes"
        
        # Code comment
        try_list = [
            # Code comment
            lambda: (pickle.load(gzip.open(filepath, 'rb')), "...Load"),
            
            # Code comment
            lambda: (pickle.load(gzip.open(filepath, 'rb'), protocol=PICKLE_PROTOCOL), f"...{PICKLE_PROTOCOL}"),
            
            # Code comment
            lambda: (pickle.load(gzip.open(filepath, 'rb'), encoding='latin-1'), "latin-1..."),
            
            # Code comment
            lambda: (pickle.load(gzip.open(filepath, 'rb'), encoding='bytes'), "bytes..."),
            
            # Code comment
            lambda: (
                pickle.loads(gzip.open(filepath, 'rb').read()), 
                "...Load"
            ),
        ]
        
        for i, load_func in enumerate(try_list):
            try:
                data, method = load_func()
                
                # Code comment
                if data is not None:
                    if isinstance(data, (list, dict, tuple, set)):
                        log_message(f"SuccessfullyLoad {filename}: ...{method}, ...: {type(data).__name__}", to_console=False)
                        return data, ""
                    else:
                        # Code comment
                        try:
                            data_list = list(data)
                            log_message(f"SuccessfullyLoad {filename}: ...{method}, ... columns..., ...: {len(data_list)}", to_console=False)
                            return data_list, ""
                        except:
                            return None, f"...: {type(data).__name__}"
            except Exception as e:
                if i == len(try_list) - 1:  # Code comment
                    return None, f"...Load...Failed: {str(e)}"
                continue
        
        return None, "...Error"
        
    except Exception as e:
        return None, f"LoadFailed: {str(e)}"

def merge_gz_files_streaming():
    """...Mergeedge_part_XXX.gz..."""
    
    # Code comment
    log_message("Start...edge_part_XXX.gz......")
    all_files = get_gz_files()
    
    total_files = len(all_files)
    log_message(f"Found {total_files}  itemsgz...")
    
    if total_files == 0:
        log_message("...Foundedge_part_XXX.gz...")
        return 0, 0, 0
    
    start_time = time.time()
    total_edges = 0
    empty_files = 0
    error_files = 0
    processed_files = 0
    
      # Comment
    temp_dir = os.path.join(SOURCE_FOLDER, "temp_merge_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir)
    
    try:
          # Comment
        log_message(f"StartProcessing file...Size: {BATCH_SIZE}...")
        
        total_batches = (total_files + BATCH_SIZE - 1) // BATCH_SIZE
        current_batch_data = []
        current_batch_edges = 0
        batch_idx = 0
        temp_chunks = []
        
          # Comment
        if total_files > 0:
            log_message(f"...: {all_files[0][0]} - {all_files[-1][0]}")
        
        for idx, (file_number, filename, filepath) in enumerate(all_files):
            try:
                  # Comment
                data, error_msg = load_gz_file(filepath, filename)
                
                if data is None:
                    empty_files += 1
                    log_message(f".../Error... {filename}: {error_msg}", to_console=False)
                    continue
                
                  # Comment
                edges = []
                
                if isinstance(data, list):
                    edges = data
                elif isinstance(data, dict):
                    # Code comment
                    for key, value in data.items():
                        if isinstance(value, list):
                            for v in value:
                                edges.append((key, v))
                        else:
                            edges.append((key, value))
                elif hasattr(data, '__iter__'):
                    # Code comment
                    edges = list(data)
                else:
                    # Code comment
                    edges = [data]
                
                edge_count = len(edges)
                
                if edge_count == 0:
                    empty_files += 1
                    log_message(f"... {filename}", to_console=False)
                    continue
                
                  # Comment
                current_batch_data.extend(edges)
                current_batch_edges += edge_count
                total_edges += edge_count
                
                  # Comment
                if (len(current_batch_data) >= MAX_EDGES_PER_CHUNK or 
                    (idx + 1) % BATCH_SIZE == 0 or 
                    idx == total_files - 1):
                    
                    if current_batch_data:
                        # SaveTemp file
                        temp_file = os.path.join(temp_dir, f"chunk_{batch_idx:05d}.pkl")
                        
                        # Code comment
                        with open(temp_file, 'wb') as f:
                            pickle.dump(current_batch_data, f, protocol=PICKLE_PROTOCOL)
                        
                        temp_chunks.append(temp_file)
                        batch_idx += 1
                        
                        log_message(f"SaveTemp file chunk_{batch_idx:05d}.pkl: {len(current_batch_data):,d} ...")
                        
                          # Comment
                        current_batch_data = []
                        gc.collect()
                
                processed_files += 1
                
                  # Comment
                if (idx + 1) % 50 == 0 or idx == total_files - 1:
                    elapsed = time.time() - start_time
                    progress = (idx + 1) / total_files * 100
                    
                    # Code comment
                    if idx > 0:
                        time_per_file = elapsed / (idx + 1)
                        remaining_files = total_files - (idx + 1)
                        remaining_time = time_per_file * remaining_files
                        time_str = f"...: {remaining_time/60:.1f}min"
                    else:
                        time_str = ""
                    
                    log_message(f"Progress: {idx + 1}/{total_files} ({progress:.1f}%), "
                               f"...: {total_edges:,d}, ...: {elapsed:.1f}s {time_str}")
                    
            except Exception as e:
                error_files += 1
                log_message(f"Processing file {filename} ...: {str(e)[:200]}", to_console=False)
                processed_files += 1
        
          # Comment
        log_message(f"\nStart merging {len(temp_chunks)}  itemsTemp file......")
        
        if len(temp_chunks) == 0:
            log_message("...Merge")
              # Comment
            try:
                os.rmdir(temp_dir)
            except:
                pass
            return 0, empty_files, error_files
        
        merge_start_time = time.time()
        
          # Comment
        with gzip.open(OUTPUT_FILE, 'wb', compresslevel=1) as out_f:
              # Comment
            final_data = []
            
            for chunk_idx, chunk_file in enumerate(temp_chunks):
                try:
                    # ReadTemp file
                    with open(chunk_file, 'rb') as f:
                        chunk_data = pickle.load(f)
                    
                      # Comment
                    final_data.extend(chunk_data)
                    
                    # Code comment
                    if len(final_data) >= 1000000:  # Code comment
                        pickle.dump(final_data, out_f, protocol=PICKLE_PROTOCOL)
                        final_data = []
                        gc.collect()
                    
                    # DisplayProgress
                    if (chunk_idx + 1) % 10 == 0:
                        progress = (chunk_idx + 1) / len(temp_chunks) * 100
                        log_message(f"MergeProgress: {chunk_idx + 1}/{len(temp_chunks)} ({progress:.1f}%)")
                    
                    # DeleteTemp file
                    try:
                        os.remove(chunk_file)
                    except:
                        pass
                    
                except Exception as e:
                    log_message(f"ProcessTemp file {chunk_file} ...: {e}")
            
              # Comment
            if final_data:
                pickle.dump(final_data, out_f, protocol=PICKLE_PROTOCOL)
        
          # Comment
        try:
            os.rmdir(temp_dir)
        except:
            # Code comment
            for file in os.listdir(temp_dir):
                try:
                    os.remove(os.path.join(temp_dir, file))
                except:
                    pass
            try:
                os.rmdir(temp_dir)
            except:
                pass
        
        merge_time = time.time() - merge_start_time
        total_time = time.time() - start_time
        
          # Comment
        log_message("\n......")
        try:
            with gzip.open(OUTPUT_FILE, 'rb') as f:
                # Code comment
                final_data_check = pickle.load(f)
                final_edge_count = len(final_data_check)
                
                # Code comment
                if final_edge_count == total_edges:
                    log_message(f"✓ ...: ... {final_edge_count:,d} ...")
                else:
                    log_message(f"⚠ Warning: ...({final_edge_count:,d})...Statistics...({total_edges:,d})...")
                
                  # Comment
                if final_edge_count > 0:
                    sample = final_data_check[0]
                    log_message(f"...: {type(sample).__name__}")
                    
                    # Code comment
                    if isinstance(sample, (tuple, list)) and len(sample) > 0:
                        log_message(f"...: {len(sample)} items..., ...: {sample}")
                
                del final_data_check
                
        except Exception as e:
            log_message(f"✗ ...Failed: {e}")
            # Code comment
            try:
                file_size = os.path.getsize(OUTPUT_FILE)
                log_message(f"...Size: {file_size} bytes ({file_size/1024/1024:.2f} MB)")
            except:
                pass
        
          # Comment
        log_message("\n" + "="*60)
        log_message("Merge complete...")
        log_message("="*60)
        log_message(f"...: {total_files}")
        log_message(f"SuccessfullyProcess: {processed_files - empty_files - error_files}")
        log_message(f"Empty file...: {empty_files}")
        log_message(f"ErrorFile count: {error_files}")
        log_message(f"...: {total_edges:,d}")
        log_message(f"Output...: {OUTPUT_FILE}")
        
        try:
            output_size = os.path.getsize(OUTPUT_FILE)
            log_message(f"Output...Size: {output_size:,d} bytes ({output_size/1024/1024:.2f} MB)")
            if total_edges > 0:
                avg_edge_size = output_size / total_edges
                log_message(f"...: {avg_edge_size:.2f} bytes")
        except:
            pass
        
        log_message(f"Process...: {total_time:.1f} s ({total_time/60:.1f} min)")
        log_message(f"Process...: {total_files/total_time:.1f} .../s")
        if total_edges > 0:
            log_message(f"Process...: {total_edges/total_time:.0f} .../s")
        log_message(f"Temp file...: {len(temp_chunks)}")
        log_message(f"Merge...: {merge_time:.1f} s")
        log_message("="*60)
        
        return total_edges, empty_files, error_files
        
    except Exception as e:
        log_message(f"Merge...Error: {e}")
        import traceback
        traceback.print_exc()
        
          # Comment
        try:
            for file in os.listdir(temp_dir):
                os.remove(os.path.join(temp_dir, file))
            os.rmdir(temp_dir)
        except:
            pass
        
        return 0, empty_files, error_files

def merge_gz_files_direct():
    """Direct merge method, suitable for small data volume cases."""
    log_message("...Merge......")
    
    # Code comment
    all_files = get_gz_files()
    total_files = len(all_files)
    
    if total_files == 0:
        log_message("...Foundedge_part_XXX.gz...")
        return 0, 0, 0
    
    start_time = time.time()
    all_edges = []
    empty_count = 0
    error_count = 0
    
    log_message(f"StartProcess {total_files}  files...")
    
    for idx, (file_number, filename, filepath) in enumerate(all_files):
        try:
            data, error_msg = load_gz_file(filepath, filename)
            
            if data is None:
                empty_count += 1
                continue
            
            # Code comment
            edges = []
            if isinstance(data, list):
                edges = data
            elif isinstance(data, dict):
                for key, value in data.items():
                    if isinstance(value, list):
                        for v in value:
                            edges.append((key, v))
                    else:
                        edges.append((key, value))
            elif hasattr(data, '__iter__'):
                edges = list(data)
            else:
                edges = [data]
            
            if edges:
                all_edges.extend(edges)
            else:
                empty_count += 1
                
        except Exception as e:
            error_count += 1
            log_message(f"Processing file {filename} ...: {e}", to_console=False)
        
        # DisplayProgress
        if (idx + 1) % 50 == 0 or idx == total_files - 1:
            elapsed = time.time() - start_time
            progress = (idx + 1) / total_files * 100
            log_message(f"Progress: {idx + 1}/{total_files} ({progress:.1f}%), "
                       f"...: {len(all_edges):,d}, ...: {elapsed:.1f}s")
    
      # Comment
    log_message(f"\nWrite... {len(all_edges):,d} ......")
    
    try:
        with gzip.open(OUTPUT_FILE, 'wb', compresslevel=1) as f:
            pickle.dump(all_edges, f, protocol=PICKLE_PROTOCOL)
        
        log_message(f"SuccessfullyWrite {OUTPUT_FILE}")
        
        # Code comment
        with gzip.open(OUTPUT_FILE, 'rb') as f:
            final_data = pickle.load(f)
            log_message(f"...: {len(final_data):,d} ...")
            del final_data
            
    except Exception as e:
        log_message(f"WriteFailed: {e}")
        return 0, empty_count, error_count
    
    total_time = time.time() - start_time
    log_message(f"Done! ...: {total_time:.1f}s")
    
    return len(all_edges), empty_count, error_count

  # Comment
if __name__ == "__main__":
      # Comment
    log_message("="*60)
    log_message("EDGE_PART_GZ...Merge...")
    log_message(f"Python...: {sys.version}")
    log_message(f"Pickle...: {PICKLE_PROTOCOL}")
    log_message(f"...: {SOURCE_FOLDER}")
    log_message(f"Output...: {OUTPUT_FILE}")
    log_message("="*60)
    
      # Comment
    if not os.path.exists(SOURCE_FOLDER):
        log_message(f"Error: ...does not exist - {SOURCE_FOLDER}")
        sys.exit(1)
    
    # Code comment
    output_dir = os.path.dirname(OUTPUT_FILE)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Code comment
    log_message("\n...Merge...:")
    log_message("1. ...Process (...)")
    log_message("2. ...Merge (...Data volume...)")
    
    try:
        choice = input("...Input... (1 ... 2...1): ").strip()
        if choice == "":
            choice = "1"
    except:
        choice = "1"
    
    if choice == "2":
        log_message("...Merge......")
        total_edges, empty_files, error_files = merge_gz_files_direct()
    else:
        log_message("...Process...")
        total_edges, empty_files, error_files = merge_gz_files_streaming()
    
    # Code comment
    log_message("\n" + "="*60)
    if total_edges > 0:
        log_message(f"✓ MergeSuccessfully! ...Merge {total_edges:,d} ...")
        log_message(f"✓ Output...: {OUTPUT_FILE}")
        
          # Comment
        try:
            file_size = os.path.getsize(OUTPUT_FILE)
            log_message(f"✓ ...Size: {file_size/1024/1024:.2f} MB")
        except:
            pass
    else:
        log_message("⚠ Merge complete...Generate...")
    
    log_message("="*60)
    
    # Code comment
    if sys.platform == "win32":
        try:
            input("\n...Enter...Exit...")
        except:
            pass

In [ ]:
import gzip
import os
import time
from datetime import datetime
import pickle
import gc
import sys
import re
import json

# ========== Configuration ==========
SOURCE_FOLDER = r"./data/entities_pair"
OUTPUT_FILE = r"./data/output\merged_edgess.pkl.gz" 
LOG_FOLDER = r"./logs"
CHECKPOINT_FOLDER = r"./checkpoints"  # Code comment

# Processing parameters
BATCH_SIZE = 100  # Code comment
MAX_EDGES_PER_CHUNK = 50000  # Code comment
PICKLE_PROTOCOL = pickle.HIGHEST_PROTOCOL  # Code comment
CHECKPOINT_INTERVAL = 50  # Code comment
MAX_BATCH_FILES_PER_TEMP = 100  # Code comment

# ========== Initialization ==========
if not os.path.exists(LOG_FOLDER):
    os.makedirs(LOG_FOLDER)

if not os.path.exists(CHECKPOINT_FOLDER):
    os.makedirs(CHECKPOINT_FOLDER)

log_file = os.path.join(LOG_FOLDER, f'log_merge_concept_pairs_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt')

def log_message(message, to_console=True, to_file=True):
    """..."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    full_message = f"[{timestamp}] {message}"
    
    if to_console:
        print(full_message)
    
    if to_file:
        with open(log_file, 'a', encoding='utf-8') as f:
            f.write(full_message + '\n')

def get_gz_files():
    """...temp_XXX...batch_XXXXX.pkl.gz...Sort... itemstemp...MAX_BATCH_FILES_PER_TEMP files"""
    pattern = re.compile(r'temp_(\d+)$')  # Code comment
    batch_pattern = re.compile(r'batch_(\d+)\.pkl\.gz$')  # Code comment
    
    all_batch_files = []
    temp_folders_info = []
    
    # Code comment
    for item in os.listdir(SOURCE_FOLDER):
        folder_path = os.path.join(SOURCE_FOLDER, item)
        match = pattern.match(item)
        
        if match and os.path.isdir(folder_path):
            temp_number = int(match.group(1))
            
            # Code comment
            batch_files_in_temp = []
            for filename in os.listdir(folder_path):
                batch_match = batch_pattern.match(filename)
                if batch_match:
                    batch_number = int(batch_match.group(1))
                    filepath = os.path.join(folder_path, filename)
                    batch_files_in_temp.append({
                        'temp_num': temp_number,
                        'batch_num': batch_number,
                        'filename': filename,
                        'filepath': filepath,
                        'sort_key': batch_number
                    })
            
            # Code comment
            batch_files_in_temp.sort(key=lambda x: x['sort_key'])
            
            # Code comment
            original_count = len(batch_files_in_temp)
            if original_count > MAX_BATCH_FILES_PER_TEMP:
                batch_files_in_temp = batch_files_in_temp[:MAX_BATCH_FILES_PER_TEMP]
                log_message(f"temp_{temp_number}: ... {original_count}  itemsbatch... {MAX_BATCH_FILES_PER_TEMP}  items (batch_0 ... batch_{MAX_BATCH_FILES_PER_TEMP-1})")
            
              # Comment
            for batch_file in batch_files_in_temp:
                all_batch_files.append(batch_file)
            
            temp_folders_info.append({
                'temp_num': temp_number,
                'total_files': original_count,
                'selected_files': len(batch_files_in_temp)
            })
    
    # Code comment
    all_batch_files.sort(key=lambda x: (x['temp_num'], x['batch_num']))
    
      # Comment
    log_message(f"\n...temp...Statistics:")
    for info in temp_folders_info:
        log_message(f"  temp_{info['temp_num']}: {info['selected_files']}/{info['total_files']}  files...")
    
    log_message(f"\n... {len(all_batch_files)}  itemsbatch_XXXXX.pkl.gz...")
    if len(all_batch_files) > 0:
        log_message(f"temp...: {all_batch_files[0]['temp_num']} - {all_batch_files[-1]['temp_num']}")
        log_message(f"batch...: {all_batch_files[0]['batch_num']} - {all_batch_files[-1]['batch_num']}")
    
    return all_batch_files

def save_checkpoint(checkpoint_file, processed_files, total_edges, current_batch_data, 
                   batch_idx, temp_chunks, completed_files_list):
    """Save..."""
    try:
          # Comment
        temp_checkpoint = checkpoint_file + '.tmp'
        
        checkpoint_data = {
            'processed_files': processed_files,
            'total_edges': total_edges,
            'batch_idx': batch_idx,
            'temp_chunks': temp_chunks,
            'completed_files': completed_files_list,
            'timestamp': datetime.now().isoformat(),
            'output_file': OUTPUT_FILE
        }
        
          # Comment
        if current_batch_data and len(current_batch_data) > 0:
            batch_data_file = checkpoint_file.replace('.json', '_batch_data.pkl')
            with open(batch_data_file, 'wb') as f:
                pickle.dump(current_batch_data, f, protocol=PICKLE_PROTOCOL)
            checkpoint_data['batch_data_file'] = batch_data_file
        else:
            checkpoint_data['batch_data_file'] = None
        
        with open(temp_checkpoint, 'w', encoding='utf-8') as f:
            json.dump(checkpoint_data, f, indent=2, ensure_ascii=False)
        
        # Code comment
        os.replace(temp_checkpoint, checkpoint_file)
        
        log_message(f"...Save: Processed {processed_files}  files, {total_edges:,d} ...", to_console=False)
        return True
    except Exception as e:
        log_message(f"Save...Failed: {e}", to_console=False)
        return False

def load_checkpoint(checkpoint_file):
    """Load..."""
    if not os.path.exists(checkpoint_file):
        return None
    
    try:
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            checkpoint_data = json.load(f)
        
          # Comment
        if checkpoint_data.get('batch_data_file') and os.path.exists(checkpoint_data['batch_data_file']):
            with open(checkpoint_data['batch_data_file'], 'rb') as f:
                checkpoint_data['current_batch_data'] = pickle.load(f)
        else:
            checkpoint_data['current_batch_data'] = []
        
        log_message(f"Load...: Processed {checkpoint_data['processed_files']}  files, "
                   f"{checkpoint_data['total_edges']:,d} ...")
        return checkpoint_data
    except Exception as e:
        log_message(f"Load...Failed: {e}")
        return None

def load_gz_file(filepath, filename=""):
    """...Loadgz...pickle..."""
    try:
        file_size = os.path.getsize(filepath)
        if file_size == 0:
            return None, f"Empty file: {file_size}bytes"
        
        # Code comment
        try_list = [
            # Code comment
            lambda: (pickle.load(gzip.open(filepath, 'rb')), "...Load"),
            
            # Code comment
            lambda: (pickle.load(gzip.open(filepath, 'rb'), protocol=PICKLE_PROTOCOL), f"...{PICKLE_PROTOCOL}"),
            
            # Code comment
            lambda: (pickle.load(gzip.open(filepath, 'rb'), encoding='latin-1'), "latin-1..."),
            
            # Code comment
            lambda: (pickle.load(gzip.open(filepath, 'rb'), encoding='bytes'), "bytes..."),
            
            # Code comment
            lambda: (
                pickle.loads(gzip.open(filepath, 'rb').read()), 
                "...Load"
            ),
        ]
        
        for i, load_func in enumerate(try_list):
            try:
                data, method = load_func()
                
                # Code comment
                if data is not None:
                    if isinstance(data, (list, dict, tuple, set)):
                        log_message(f"SuccessfullyLoad {filename}: ...{method}, ...: {type(data).__name__}", to_console=False)
                        return data, ""
                    else:
                        # Code comment
                        try:
                            data_list = list(data)
                            log_message(f"SuccessfullyLoad {filename}: ...{method}, ... columns..., ...: {len(data_list)}", to_console=False)
                            return data_list, ""
                        except:
                            return None, f"...: {type(data).__name__}"
            except Exception as e:
                if i == len(try_list) - 1:  # Code comment
                    return None, f"...Load...Failed: {str(e)}"
                continue
        
        return None, "...Error"
        
    except Exception as e:
        return None, f"LoadFailed: {str(e)}"

def merge_gz_files_streaming(resume=True):
    """...Mergetemp_XXX...batch_XXXXX.pkl.gz [description]  itemstemp...ProcessMAX_BATCH_FILES_PER_TEMP files"""
    
    # Code comment
    log_message("Start...temp_XXX...batch_XXXXX.pkl.gz......")
    all_files = get_gz_files()
    
    total_files = len(all_files)
    log_message(f"Found {total_files}  itemsbatch... itemstemp...{MAX_BATCH_FILES_PER_TEMP} items...")
    
    if total_files == 0:
        log_message("...Foundbatch_XXXXX.pkl.gz...")
        return 0, 0, 0
    
      # Comment
    task_id = re.sub(r'[^\w\-_]', '_', OUTPUT_FILE)
    checkpoint_file = os.path.join(CHECKPOINT_FOLDER, f'checkpoint_{task_id}.json')
    
      # Comment
    processed_files_count = 0
    total_edges = 0
    current_batch_data = []
    batch_idx = 0
    temp_chunks = []
    completed_files_set = set()
    
    if resume:
        checkpoint = load_checkpoint(checkpoint_file)
        if checkpoint:
            processed_files_count = checkpoint['processed_files']
            total_edges = checkpoint['total_edges']
            batch_idx = checkpoint['batch_idx']
            temp_chunks = checkpoint['temp_chunks']
            completed_files_set = set(checkpoint['completed_files'])
            current_batch_data = checkpoint.get('current_batch_data', [])
            
            log_message(f"...: Processed {processed_files_count}/{total_files}  files")
            
            # Code comment
            existing_chunks = []
            for chunk in temp_chunks:
                if os.path.exists(chunk):
                    existing_chunks.append(chunk)
                else:
                    log_message(f"Warning: Temp filedoes not exist {chunk}", to_console=False)
            temp_chunks = existing_chunks
            batch_idx = len(temp_chunks)
    
    start_time = time.time()
    empty_files = 0
    error_files = 0
    processed_files = 0
    
      # Comment
    temp_dir = os.path.join(SOURCE_FOLDER, "temp_merge_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir)
    
      # Comment
    if temp_chunks:
        log_message(f"Move {len(temp_chunks)}  items...Temp file......")
        new_temp_chunks = []
        for old_chunk in temp_chunks:
            if os.path.exists(old_chunk):
                new_chunk = os.path.join(temp_dir, os.path.basename(old_chunk))
                os.rename(old_chunk, new_chunk)
                new_temp_chunks.append(new_chunk)
        temp_chunks = new_temp_chunks
    
    try:
          # Comment
        log_message(f"StartProcessing file...Size: {BATCH_SIZE}...")
        
        total_batches = (total_files + BATCH_SIZE - 1) // BATCH_SIZE
        
          # Comment
        if total_files > 0:
            log_message(f"temp...: {all_files[0]['temp_num']} - {all_files[-1]['temp_num']}")
            log_message(f"batch...: {all_files[0]['batch_num']} - {all_files[-1]['batch_num']}")
        
        for idx, file_info in enumerate(all_files):
              # Comment
            if idx < processed_files_count:
                continue
            
              # Comment
            file_key = f"{file_info['temp_num']}_{file_info['batch_num']}"
            if file_key in completed_files_set:
                log_message(f"Skip...Processing file: {file_info['filename']}", to_console=False)
                processed_files += 1
                continue
            
            try:
                filename = file_info['filename']
                filepath = file_info['filepath']
                temp_num = file_info['temp_num']
                batch_num = file_info['batch_num']
                
                  # Comment
                data, error_msg = load_gz_file(filepath, filename)
                
                if data is None:
                    empty_files += 1
                    log_message(f".../Error... temp_{temp_num}/{filename}: {error_msg}", to_console=False)
                    completed_files_set.add(file_key)
                    processed_files += 1
                    continue
                
                  # Comment
                edges = []
                
                if isinstance(data, list):
                    edges = data
                elif isinstance(data, dict):
                    # Code comment
                    for key, value in data.items():
                        if isinstance(value, list):
                            for v in value:
                                edges.append((key, v))
                        else:
                            edges.append((key, value))
                elif hasattr(data, '__iter__'):
                    # Code comment
                    edges = list(data)
                else:
                    # Code comment
                    edges = [data]
                
                edge_count = len(edges)
                
                if edge_count == 0:
                    empty_files += 1
                    log_message(f"... temp_{temp_num}/{filename}", to_console=False)
                    completed_files_set.add(file_key)
                    processed_files += 1
                    continue
                
                  # Comment
                current_batch_data.extend(edges)
                current_batch_edges = edge_count
                total_edges += edge_count
                
                  # Comment
                if (len(current_batch_data) >= MAX_EDGES_PER_CHUNK or 
                    (idx + 1) % BATCH_SIZE == 0 or 
                    idx == total_files - 1):
                    
                    if current_batch_data:
                        # SaveTemp file
                        temp_file = os.path.join(temp_dir, f"chunk_{batch_idx:05d}.pkl")
                        
                        # Code comment
                        with open(temp_file, 'wb') as f:
                            pickle.dump(current_batch_data, f, protocol=PICKLE_PROTOCOL)
                        
                        temp_chunks.append(temp_file)
                        batch_idx += 1
                        
                        log_message(f"SaveTemp file chunk_{batch_idx:05d}.pkl: {len(current_batch_data):,d} ...")
                        
                          # Comment
                        current_batch_data = []
                        gc.collect()
                
                # Code comment
                completed_files_set.add(file_key)
                processed_files += 1
                
                  # Comment
                if (processed_files - processed_files_count) % CHECKPOINT_INTERVAL == 0 or processed_files == total_files:
                    save_checkpoint(checkpoint_file, processed_files, total_edges, 
                                  current_batch_data, batch_idx, temp_chunks, 
                                  list(completed_files_set))
                
                  # Comment
                if (idx + 1) % 50 == 0 or idx == total_files - 1:
                    elapsed = time.time() - start_time
                    progress = (idx + 1) / total_files * 100
                    
                    # Code comment
                    if idx > processed_files_count:
                        time_per_file = elapsed / (idx + 1 - processed_files_count)
                        remaining_files = total_files - (idx + 1)
                        remaining_time = time_per_file * remaining_files
                        time_str = f"...: {remaining_time/60:.1f}min"
                    else:
                        time_str = ""
                    
                    log_message(f"Progress: {idx + 1}/{total_files} ({progress:.1f}%), "
                               f"...: {total_edges:,d}, ...: {elapsed:.1f}s {time_str}")
                    
            except Exception as e:
                error_files += 1
                log_message(f"Processing file {file_info['filename']} ...: {str(e)[:200]}", to_console=False)
                processed_files += 1
        
          # Comment
        log_message(f"\nStart merging {len(temp_chunks)}  itemsTemp file......")
        
        if len(temp_chunks) == 0:
            log_message("...Merge")
              # Comment
            try:
                os.rmdir(temp_dir)
            except:
                pass
            return 0, empty_files, error_files
        
        merge_start_time = time.time()
        
          # Comment
        output_mode = 'ab' if resume and os.path.exists(OUTPUT_FILE) else 'wb'
        
          # Comment
        with gzip.open(OUTPUT_FILE, output_mode, compresslevel=1) as out_f:
              # Comment
            if output_mode == 'ab':
                log_message(" [description] ")
            else:
                log_message("...Create...")
            
              # Comment
            final_data = []
            processed_chunks = set()
            
            # Code comment
            if resume and os.path.exists(checkpoint_file):
                try:
                    with open(checkpoint_file, 'r', encoding='utf-8') as f:
                        merge_checkpoint = json.load(f)
                        processed_chunks = set(merge_checkpoint.get('processed_chunks', []))
                except:
                    pass
            
            for chunk_idx, chunk_file in enumerate(temp_chunks):
                  # Comment
                if chunk_file in processed_chunks:
                    log_message(f"SkipProcessedTemp file: {os.path.basename(chunk_file)}", to_console=False)
                    continue
                    
                try:
                    # ReadTemp file
                    with open(chunk_file, 'rb') as f:
                        chunk_data = pickle.load(f)
                    
                      # Comment
                    final_data.extend(chunk_data)
                    
                    # Code comment
                    if len(final_data) >= 1000000:  # Code comment
                        pickle.dump(final_data, out_f, protocol=PICKLE_PROTOCOL)
                        final_data = []
                        gc.collect()
                    
                    # Code comment
                    processed_chunks.add(chunk_file)
                    
                      # Comment
                    merge_checkpoint = {
                        'processed_chunks': list(processed_chunks),
                        'total_chunks': len(temp_chunks),
                        'timestamp': datetime.now().isoformat()
                    }
                    with open(checkpoint_file + '_merge.json', 'w', encoding='utf-8') as f:
                        json.dump(merge_checkpoint, f, indent=2)
                    
                    # DisplayProgress
                    if (chunk_idx + 1) % 10 == 0 or chunk_idx == len(temp_chunks) - 1:
                        progress = (chunk_idx + 1) / len(temp_chunks) * 100
                        log_message(f"MergeProgress: {chunk_idx + 1}/{len(temp_chunks)} ({progress:.1f}%)")
                    
                    # DeleteTemp file
                    try:
                        os.remove(chunk_file)
                    except:
                        pass
                    
                except Exception as e:
                    log_message(f"ProcessTemp file {chunk_file} ...: {e}")
            
              # Comment
            if final_data:
                pickle.dump(final_data, out_f, protocol=PICKLE_PROTOCOL)
        
          # Comment
        try:
            os.rmdir(temp_dir)
        except:
            # Code comment
            for file in os.listdir(temp_dir):
                try:
                    os.remove(os.path.join(temp_dir, file))
                except:
                    pass
            try:
                os.rmdir(temp_dir)
            except:
                pass
        
          # Comment
        try:
            os.remove(checkpoint_file)
            merge_checkpoint_file = checkpoint_file + '_merge.json'
            if os.path.exists(merge_checkpoint_file):
                os.remove(merge_checkpoint_file)
        except:
            pass
        
        merge_time = time.time() - merge_start_time
        total_time = time.time() - start_time
        
          # Comment
        log_message("\n......")
        try:
            with gzip.open(OUTPUT_FILE, 'rb') as f:
                # Code comment
                final_data_check = pickle.load(f)
                final_edge_count = len(final_data_check)
                
                # Code comment
                if final_edge_count == total_edges:
                    log_message(f"✓ ...: ... {final_edge_count:,d} ...")
                else:
                    log_message(f"⚠ Warning: ...({final_edge_count:,d})...Statistics...({total_edges:,d})...")
                
                  # Comment
                if final_edge_count > 0:
                    sample = final_data_check[0]
                    log_message(f"...: {type(sample).__name__}")
                    
                    # Code comment
                    if isinstance(sample, (tuple, list)) and len(sample) > 0:
                        log_message(f"...: {len(sample)} items..., ...: {sample}")
                
                del final_data_check
                
        except Exception as e:
            log_message(f"✗ ...Failed: {e}")
            # Code comment
            try:
                file_size = os.path.getsize(OUTPUT_FILE)
                log_message(f"...Size: {file_size} bytes ({file_size/1024/1024:.2f} MB)")
            except:
                pass
        
          # Comment
        log_message("\n" + "="*60)
        log_message("Merge complete...")
        log_message("="*60)
        log_message(f"...: {total_files}")
        log_message(f"SuccessfullyProcess: {processed_files - empty_files - error_files}")
        log_message(f"Empty file...: {empty_files}")
        log_message(f"ErrorFile count: {error_files}")
        log_message(f"...: {total_edges:,d}")
        log_message(f"Output...: {OUTPUT_FILE}")
        log_message(f"... itemstemp...Processing file...: {MAX_BATCH_FILES_PER_TEMP}")
        
        try:
            output_size = os.path.getsize(OUTPUT_FILE)
            log_message(f"Output...Size: {output_size:,d} bytes ({output_size/1024/1024:.2f} MB)")
            if total_edges > 0:
                avg_edge_size = output_size / total_edges
                log_message(f"...: {avg_edge_size:.2f} bytes")
        except:
            pass
        
        log_message(f"Process...: {total_time:.1f} s ({total_time/60:.1f} min)")
        log_message(f"Process...: {total_files/total_time:.1f} .../s")
        if total_edges > 0:
            log_message(f"Process...: {total_edges/total_time:.0f} .../s")
        log_message(f"Temp file...: {len(temp_chunks)}")
        log_message(f"Merge...: {merge_time:.1f} s")
        log_message("="*60)
        
        return total_edges, empty_files, error_files
        
    except Exception as e:
        log_message(f"Merge...Error: {e}")
        import traceback
        traceback.print_exc()
        
          # Comment
        save_checkpoint(checkpoint_file, processed_files, total_edges, 
                       current_batch_data, batch_idx, temp_chunks, 
                       list(completed_files_set))
        
          # Comment
        # Note: Do not clear temp files, for reuse during resume.
        
        return 0, empty_files, error_files

def merge_gz_files_direct(resume=True):
    """Direct merge method, supports resume, suitable for data volume... itemstemp...ProcessMAX_BATCH_FILES_PER_TEMP files"""
    log_message("...Merge......")
    
    # Code comment
    all_files = get_gz_files()
    total_files = len(all_files)
    
    if total_files == 0:
        log_message("...Foundbatch_XXXXX.pkl.gz...")
        return 0, 0, 0
    
      # Comment
    task_id = re.sub(r'[^\w\-_]', '_', OUTPUT_FILE)
    checkpoint_file = os.path.join(CHECKPOINT_FOLDER, f'direct_checkpoint_{task_id}.json')
    
      # Comment
    processed_files_count = 0
    completed_files_set = set()
    
    if resume:
        if os.path.exists(checkpoint_file):
            try:
                with open(checkpoint_file, 'r', encoding='utf-8') as f:
                    checkpoint = json.load(f)
                    processed_files_count = checkpoint.get('processed_files', 0)
                    completed_files_set = set(checkpoint.get('completed_files', []))
                    log_message(f"...: Processed {processed_files_count}/{total_files}  files")
            except Exception as e:
                log_message(f"Load...Failed: {e}")
    
    start_time = time.time()
    all_edges = []
    empty_count = 0
    error_count = 0
    
    # Code comment
    if resume and os.path.exists(OUTPUT_FILE) and processed_files_count > 0:
        try:
            with gzip.open(OUTPUT_FILE, 'rb') as f:
                all_edges = pickle.load(f)
                log_message(f"Load...Output...: {len(all_edges):,d} ...")
        except Exception as e:
            log_message(f"Load...Output...Failed: {e}...Start")
            all_edges = []
            processed_files_count = 0
            completed_files_set.clear()
    
    log_message(f"StartProcess {total_files}  files...")
    log_message(f"... itemstemp...Process {MAX_BATCH_FILES_PER_TEMP}  files")
    
    for idx, file_info in enumerate(all_files):
          # Comment
        if idx < processed_files_count:
            continue
        
        file_key = f"{file_info['temp_num']}_{file_info['batch_num']}"
        if file_key in completed_files_set:
            log_message(f"Skip...Processing file: {file_info['filename']}", to_console=False)
            continue
        
        try:
            filename = file_info['filename']
            filepath = file_info['filepath']
            temp_num = file_info['temp_num']
            batch_num = file_info['batch_num']
            
            data, error_msg = load_gz_file(filepath, filename)
            
            if data is None:
                empty_count += 1
                completed_files_set.add(file_key)
                continue
            
            # Code comment
            edges = []
            if isinstance(data, list):
                edges = data
            elif isinstance(data, dict):
                for key, value in data.items():
                    if isinstance(value, list):
                        for v in value:
                            edges.append((key, v))
                    else:
                        edges.append((key, value))
            elif hasattr(data, '__iter__'):
                edges = list(data)
            else:
                edges = [data]
            
            if edges:
                all_edges.extend(edges)
            else:
                empty_count += 1
            
            completed_files_set.add(file_key)
            
              # Comment
            if len(completed_files_set) % CHECKPOINT_INTERVAL == 0:
                checkpoint_data = {
                    'processed_files': idx + 1,
                    'completed_files': list(completed_files_set),
                    'timestamp': datetime.now().isoformat()
                }
                with open(checkpoint_file, 'w', encoding='utf-8') as f:
                    json.dump(checkpoint_data, f, indent=2)
                
                # Code comment
                with gzip.open(OUTPUT_FILE, 'wb', compresslevel=1) as f:
                    pickle.dump(all_edges, f, protocol=PICKLE_PROTOCOL)
                
                log_message(f"...Save: Processed {idx + 1}/{total_files}  files, {len(all_edges):,d} ...")
                
        except Exception as e:
            error_count += 1
            log_message(f"Processing file {file_info['filename']} ...: {e}", to_console=False)
        
        # DisplayProgress
        if (idx + 1) % 50 == 0 or idx == total_files - 1:
            elapsed = time.time() - start_time
            progress = (idx + 1) / total_files * 100
            log_message(f"Progress: {idx + 1}/{total_files} ({progress:.1f}%), "
                       f"...: {len(all_edges):,d}, ...: {elapsed:.1f}s")
    
      # Comment
    log_message(f"\nWrite... {len(all_edges):,d} ......")
    
    try:
        with gzip.open(OUTPUT_FILE, 'wb', compresslevel=1) as f:
            pickle.dump(all_edges, f, protocol=PICKLE_PROTOCOL)
        
        log_message(f"SuccessfullyWrite {OUTPUT_FILE}")
        
          # Comment
        try:
            os.remove(checkpoint_file)
        except:
            pass
        
        # Code comment
        with gzip.open(OUTPUT_FILE, 'rb') as f:
            final_data = pickle.load(f)
            log_message(f"...: {len(final_data):,d} ...")
            del final_data
            
    except Exception as e:
        log_message(f"WriteFailed: {e}")
        return 0, empty_count, error_count
    
    total_time = time.time() - start_time
    log_message(f"Done! ...: {total_time:.1f}s")
    
    return len(all_edges), empty_count, error_count

  # Comment
if __name__ == "__main__":
      # Comment
    log_message("="*60)
    log_message("BATCH...Merge... (...)")
    log_message(f"Python...: {sys.version}")
    log_message(f"Pickle...: {PICKLE_PROTOCOL}")
    log_message(f"...: {SOURCE_FOLDER}")
    log_message(f"Output...: {OUTPUT_FILE}")
    log_message(f"...: {CHECKPOINT_FOLDER}")
    log_message(f"... itemstemp...Process: {MAX_BATCH_FILES_PER_TEMP}  itemsbatch...")
    log_message("="*60)
    
      # Comment
    if not os.path.exists(SOURCE_FOLDER):
        log_message(f"Error: ...does not exist - {SOURCE_FOLDER}")
        sys.exit(1)
    
    # Code comment
    output_dir = os.path.dirname(OUTPUT_FILE)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
      # Comment
    task_id = re.sub(r'[^\w\-_]', '_', OUTPUT_FILE)
    checkpoint_file = os.path.join(CHECKPOINT_FOLDER, f'checkpoint_{task_id}.json')
    has_unfinished = os.path.exists(checkpoint_file)
    
    if has_unfinished:
        log_message("\n...Done...Merge...!")
        log_message(f"...: {checkpoint_file}")
        try:
            with open(checkpoint_file, 'r', encoding='utf-8') as f:
                checkpoint = json.load(f)
                log_message(f"...Progress: Processed {checkpoint.get('processed_files', 0)}  files, "
                          f"{checkpoint.get('total_edges', 0):,d} ...")
        except:
            pass
        
        resume_choice = input("...Resume...? (y/n...y): ").strip().lower()
        if resume_choice == '' or resume_choice == 'y' or resume_choice == 'yes':
            resume_flag = True
            log_message("...Interrupt...ResumeProcess...")
        else:
            resume_flag = False
            log_message("...StartProcess...Ignore...Progress......")
    else:
        resume_flag = True
        log_message("...Done...Start...Merge...")
    
    # Code comment
    log_message("\n...Merge...:")
    log_message("1. ...Process ( [description] )")
    log_message("2. ...Merge (...Data volume [description] )")
    
    try:
        choice = input("...Input... (1 ... 2...1): ").strip()
        if choice == "":
            choice = "1"
    except:
        choice = "1"
    
    if choice == "2":
        log_message("...Merge......")
        total_edges, empty_files, error_files = merge_gz_files_direct(resume=resume_flag)
    else:
        log_message("...Process...")
        total_edges, empty_files, error_files = merge_gz_files_streaming(resume=resume_flag)
    
    # Code comment
    log_message("\n" + "="*60)
    if total_edges > 0:
        log_message(f"✓ MergeSuccessfully! ...Merge {total_edges:,d} ...")
        log_message(f"✓ Output...: {OUTPUT_FILE}")
        
          # Comment
        try:
            file_size = os.path.getsize(OUTPUT_FILE)
            log_message(f"✓ ...Size: {file_size/1024/1024:.2f} MB")
        except:
            pass
    else:
        log_message("⚠ Merge complete...Generate...")
    
    log_message("="*60)
    
    # Code comment
    if sys.platform == "win32":
        try:
            input("\n...Enter...Exit...")
        except:
            pass

In [ ]:
import gzip
import pickle

file_path = r"./data/entities_pair\temp_816\batch_0069.pkl.gz"

  # Comment
with gzip.open(file_path, 'rb') as f:
    data = pickle.load(f)

  # Comment
print(f"...: {type(data)}")
print(f"...: {data}")

In [ ]:
import pandas as pd
import os

# Code comment
input_file = r"./data/data_graph\final_dynamic_graph.parquet"
output_file = r"./data/data_graph\full_dynamic_graph.csv"

try:
      # Comment
    print(f"Reading file: {input_file}")
    df = pd.read_parquet(input_file)
    
      # Comment
    print(f"...: {df.shape}")
    print(f" columns...: {list(df.columns)}")
    print(f"...:\n{df.dtypes}")
    
    # Code comment
    print(f"...CSV: {output_file}")
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"...Done...Save...: {output_file}")
    
      # Comment
    input_size = os.path.getsize(input_file) / (1024 * 1024)  # Code comment
    output_size = os.path.getsize(output_file) / (1024 * 1024)
    print(f"...Size: {input_size:.2f} MB")
    print(f"CSV...Size: {output_size:.2f} MB")
    
except FileNotFoundError:
    print(f"Error... {input_file}")
except Exception as e:
    print(f"...Error: {str(e)}")

In [ ]:
import pandas as pd

  # Comment
df1 = pd.read_parquet(r"./data/data_concept_graph\full_dynamic_graphs.parquet")
df2 = pd.read_parquet(r"./data/data_concept_graph\full_dynamic_graph.parquet")

  # Comment
merged_df = pd.concat([df1, df2], ignore_index=True)

  # Comment
merged_df.to_parquet(r'./data/data_graph\final_dynamic_graph.parquet', index=False)

print("... files...SuccessfullyMerge")

In [ ]:
import pandas as pd

# Code comment
df = pd.read_parquet(r"./data/data_concept_graph\full_dynamic_graph.parquet", columns=['v1', 'v2'])

  # Comment
unique_count = len(pd.unique(pd.concat([df['v1'], df['v2']])))

print(f"...ID...: {unique_count}")

In [ ]:
import pyarrow.parquet as pq

  # Comment
file_path = r"./data/data_concept_graph\full_dynamic_graph.parquet"
parquet_file = pq.ParquetFile(file_path)

# Code comment
row_count = parquet_file.metadata.num_rows

print(f"... {row_count}  rows")